<a href="https://colab.research.google.com/github/denisejroth/pixels-frames-transformers/blob/main/part1_pixels/02_building_an_image_corpus/notebooks/02_building_an_image_corpus_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pixels, Frames & Transformers
## Module 02: Building an Image Corpus · Solutions

**A Self-Paced Course in Computational Image and Video Analysis**  
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every ✏️ exercise is filled in with one possible answer and a short comment. There is usually more than one good way to solve an exercise, and your numbers may differ a little from run to run, so read the comments as "what to look for" rather than exact results.

Most image research starts with a collection problem, not a modeling problem. Images are easy to
find and hard to use properly: they carry copyright, they show people who never agreed to be
studied, and platforms restrict what you can download. In this notebook you build two small
research corpora from open sources and document them so that you, a reviewer and a future
replicator can see exactly where every image came from.

In this notebook you will:

- work through the questions to ask before downloading any image: licenses, people, platform
  terms and sharing
- query the Wikimedia Commons API for photos of demonstrations and protests in the Netherlands,
  with the license and author of every file
- download images politely and record everything in a **manifest**
- collect food packaging photos from Open Food Facts
- find exact and near duplicates with perceptual hashing
- decide what you can share, and write a short datasheet for your corpus

**Data.** [Wikimedia Commons](https://commons.wikimedia.org/) hosts more than 100 million freely
licensed media files, each with a license and author on record.
[Open Food Facts](https://world.openfoodfacts.org/) is a collaborative database of food products
whose photos are licensed CC BY-SA. Both collections return in modules 03 and 04.

This notebook runs on a normal CPU runtime. The downloads take about ten minutes in total,
mostly because we deliberately pause between requests.

## 0. Setup

`imagehash` (for perceptual hashing in section 5) is the one package Colab doesn't have.

Wikimedia and Open Food Facts both ask clients to identify themselves with a descriptive
`User-Agent`: a name, a version and a way to contact whoever runs the code. Requests with a
generic or missing User-Agent can be blocked. We use the course's address here. **If you adapt
this code for your own project, put your own project name and contact details in.**

In [ ]:
!pip install -q imagehash

In [ ]:
import hashlib
import io
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import imagehash
from PIL import Image, ImageOps

HEADERS = {"User-Agent": "PixelsFramesTransformers/1.0 (https://github.com/denisejroth/pixels-frames-transformers) course-notebook"}

RETRY_STATUS = {429, 500, 502, 503, 504}     # busy or briefly unavailable: worth another try

def download(url, params=None, retries=4, wait=15):
    """GET a URL (with optional query parameters) and return the response. If the server is busy
    or briefly down, wait and try again (15, 30, 60, 120 seconds); raise an error if it still fails."""
    for attempt in range(retries + 1):
        response = requests.get(url, params=params, headers=HEADERS, timeout=60)
        if response.status_code not in RETRY_STATUS or attempt == retries:
            break
        retry_after = response.headers.get("Retry-After", "")
        delay = int(retry_after) if retry_after.isdigit() else wait * 2 ** attempt
        print(f"{urlparse(url).netloc} answered {response.status_code}; trying again in {delay} seconds")
        time.sleep(delay)
    response.raise_for_status()
    return response

CORPUS_DIR = Path("corpus")
CORPUS_DIR.mkdir(exist_ok=True)

def contact_sheet(paths, labels=None, n_cols=8, size=1.6):
    """Show image files in a grid, with an optional label under each."""
    paths = list(paths)
    labels = labels if labels is not None else [""] * len(paths)
    n_rows = max(1, int(np.ceil(len(paths) / n_cols)))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * size, n_rows * size * 1.15), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, path, label in zip(axes.flat, paths, labels):
        with Image.open(path) as img:
            img.thumbnail((300, 300))
            ax.imshow(img.convert("RGB"))
        ax.set_title(str(label)[:22], fontsize=7)
    plt.tight_layout()
    plt.show()

print("Setup complete!")

## 1. Before you download anything

It is tempting to start collecting and sort out the rules later. With images that tends to go
wrong, because the questions below decide what you are allowed to do with the corpus once you
have it. Answer them first:

1. **Who holds the rights to these images, and under what license can I use them?**
2. **Do the images show identifiable people?** If so, what is my legal basis for processing
   them, and how will I protect them?
3. **What do the terms of the platform or archive allow?** Downloading, automated access,
   research use?
4. **What will I need to share** when I publish (figures, the data, a replication package), and
   does the license allow that?

### Licenses in practice

Open licenses make the first question easy to answer. Here is what the common ones mean for a
research project. This is a starting point, not legal advice: your university's research data
office or legal team can help with your specific case.

| License | Analyze it | Show it in a paper or talk | Share the files | Conditions |
|---|---|---|---|---|
| Public domain, CC0 | yes | yes | yes | none (crediting the source is still good practice) |
| CC BY | yes | yes | yes | credit the creator, link the license, say if you changed it |
| CC BY-SA | yes | yes | yes | as CC BY, and anything you build from it (a cropped version, a dataset of the images) must carry the same license |
| CC BY-NC | yes | usually, but check whether your publisher counts as commercial | for non-commercial use | as CC BY, non-commercial only |
| CC BY-ND | yes | yes, unchanged | yes, unchanged | no adaptations; cropping or drawing boxes on it can count as one |
| All rights reserved | often, under research exceptions in your jurisdiction | only under quotation or fair use rules, case by case | generally no | ask the rights holder, or share identifiers instead of files |

In the EU, Article 3 of the Copyright in the Digital Single Market Directive (2019/790) allows
research organizations to do text and data mining, including on images, on works they have lawful
access to, for scientific research. That covers *analyzing* such images. It does not give you the
right to redistribute them.

### People in pictures

A photo of an identifiable person is **personal data** under the GDPR, whether or not you know
their name. That does not stop research: the GDPR allows processing for scientific research with
safeguards such as collecting only what you need, storing it securely and pseudonymizing where
possible. A few points deserve extra care:

- Using a face to **identify** someone (face recognition) turns it into biometric data, a special
  category with stricter rules. Analyzing a photo without identifying anyone is not.
- **Public figures in their public role** (politicians at work, like the portraits in module 01)
  are different from bystanders at a demonstration, who may not want their participation known.
- **Minors** need extra protection, also in figures you publish.
- Whatever you publish (example images in a paper, a shared dataset) reaches far more people than
  your analysis does. Blurring faces in figures is often a good idea.

Your institution's privacy officer and ethics committee are the people to talk to before you
collect images of people at scale.

> **✏️ Exercise 1**
>
> For each scenario, decide whether you can do it, under what conditions, and what you would check.
>
> 1. You found a CC BY-NC photo of a climate protest and want to use it as an example figure in a
>    journal article.
> 2. You built a dataset of 2,000 CC BY-SA images from Wikimedia Commons, cropped to the people in
>    them, and want to publish it on Zenodo under CC BY so that others can reuse it freely.
> 3. You want to train a classifier on news photos from a newspaper's website (all rights
>    reserved) for a study at your university, and publish only the model's results.
> 4. You want to show a screenshot of a TikTok video by a private user in a conference talk.

**✅ One possible answer**

1. **Probably yes, with care.** Academic publishing is usually treated as non-commercial, but some
   publishers are commercial companies and treat NC material as a problem, especially for open
   access articles that allow commercial reuse. Check your publisher's policy, credit the creator
   and link the license. If the protesters are identifiable, consider whether they would mind.
2. **Not under CC BY.** Your cropped images are adaptations of CC BY-SA works, so the dataset must
   be shared under CC BY-SA too, with credit for every image (a manifest with authors and
   licenses does this). The people in the images are still personal data, so think about whether
   a dataset of cropped faces from demonstrations should be public at all.
3. **The analysis is likely fine; the images are not shareable.** In the EU, the text and data
   mining exception for research organizations covers analyzing works you have lawful access to,
   but it does not let you redistribute them. Check the site's terms for automated access, store
   the images securely, and share URLs or identifiers plus your code instead of the files.
4. **Be careful.** Quotation rules may cover a screenshot shown for critical discussion in a talk,
   but a private user did not choose to be in your slides. Prefer blurring their face and
   username, or ask for permission, and never show minors. Platform terms also apply to how you
   obtained the video.

## 2. Where research images come from

| Source | What you get | Typical constraints |
|---|---|---|
| Archives and museums | curated collections with metadata, increasingly with open licenses | varies per collection; read the reuse terms |
| Wikimedia Commons | freely licensed images, each with author and license | content is community-curated, so categories can be messy |
| Open data portals and projects | for example Open Food Facts, government photo collections | usually open licenses with attribution |
| Platform research access | for example the YouTube Data API, the TikTok Research API, Meta's Content Library, and data access for vetted researchers under the EU Digital Services Act | mostly **metadata** (IDs, captions, counts), not the media files themselves; approval needed |
| Data donation | participants share their own data with you | consent covers the participant, not other people in their photos |
| Scraping websites | whatever is publicly visible | terms of service, copyright and privacy all apply; often a gray area |

In this notebook we use two sources that are open by design, so you can focus on the mechanics.
The habits you build here (record the source and license of everything, pause between requests,
keep a manifest) matter even more with the harder sources.

## 3. Querying Wikimedia Commons

Commons organizes files in **categories**, which contain files and further subcategories. Its
API (`https://commons.wikimedia.org/w/api.php`) lets you list the members of a category and ask
for the details of each file, including its license.

We start with a helper that sends a query to the API, and one that lists all files or all
subcategories in a category. The API returns at most 500 results per request and gives a
`continue` value when there are more, so the helper keeps asking until it has everything.

In [ ]:
COMMONS_API = "https://commons.wikimedia.org/w/api.php"

def commons_api(params):
    """Send a query to the Wikimedia Commons API and return the JSON response."""
    return download(COMMONS_API, params={"format": "json", "formatversion": 2, **params}).json()

def category_members(category, member_type="file"):
    """List the titles of all files ("file") or subcategories ("subcat") in a category."""
    params = {"action": "query", "list": "categorymembers", "cmtitle": f"Category:{category}",
              "cmtype": member_type, "cmlimit": 500}
    titles = []
    while True:
        data = commons_api(params)
        titles += [member["title"] for member in data.get("query", {}).get("categorymembers", [])]
        if "continue" not in data:
            return titles
        params.update(data["continue"])
        time.sleep(0.5)

category_members("Demonstrations and protests in the Netherlands in 2019")[:10]

Categories on Commons are maintained by volunteers, so a category tree can contain things you did
not expect. We walk the category for each recent year plus one level of subcategories, keep only
JPEG and PNG photos, and skip subcategories whose names suggest content we don't want in a
teaching corpus. **Always look at what you collected before you use it.**

In [ ]:
YEARS = range(2015, 2025)
START_CATEGORIES = [f"Demonstrations and protests in the Netherlands in {year}" for year in YEARS]
SKIP_WORDS = ["naked", "nude", "naakt", "wnbr"]

def collect_files(categories, depth=1, max_files=300):
    """Walk categories (and subcategories up to `depth` levels down); return {file title: category}."""
    found, seen = {}, set()
    queue = [(category, 0) for category in categories]
    while queue and len(found) < max_files:
        category, level = queue.pop(0)
        if category in seen or any(word in category.lower() for word in SKIP_WORDS):
            continue
        seen.add(category)
        for title in category_members(category, "file"):
            if title.lower().endswith((".jpg", ".jpeg", ".png")):
                found.setdefault(title, category)
        if level < depth:
            for sub in category_members(category, "subcat"):
                queue.append((sub.removeprefix("Category:"), level + 1))
        time.sleep(0.5)
    return dict(list(found.items())[:max_files])

protest_files = collect_files(START_CATEGORIES, depth=1, max_files=300)
print(f"Found {len(protest_files)} photos")
pd.Series(protest_files).value_counts().head(10)

Now we ask for the details of every file: the URL of a 500-pixel-wide thumbnail (plenty for
analysis, and much lighter on Wikimedia's servers than the full files), the original size, and
the **extended metadata**, which holds the license, the author and a description. Some of these
fields contain HTML, which we strip. The API accepts up to 50 titles per request.

In [ ]:
def strip_html(text):
    """Remove HTML tags and surrounding whitespace."""
    return re.sub(r"<[^>]+>", "", text or "").strip()

def meta_value(extmetadata, key):
    return strip_html(extmetadata.get(key, {}).get("value", ""))

def file_info(titles, found_in=None, thumb_width=500):
    """Look up thumbnail URL, size, license and author for a list of Commons file titles."""
    records = []
    for start in range(0, len(titles), 50):
        batch = titles[start:start + 50]
        data = commons_api({"action": "query", "titles": "|".join(batch), "prop": "imageinfo",
                            "iiprop": "url|size|mime|extmetadata", "iiurlwidth": thumb_width})
        for page in data["query"]["pages"]:
            if page.get("missing") or "imageinfo" not in page:
                continue
            info = page["imageinfo"][0]
            meta = info.get("extmetadata", {})
            records.append({
                "title": page["title"],
                "found_in": (found_in or {}).get(page["title"], ""),
                "source_url": info.get("thumburl", info["url"]),
                "page_url": info["descriptionurl"],
                "original_width": info["width"],
                "original_height": info["height"],
                "license": meta_value(meta, "LicenseShortName"),
                "license_url": meta_value(meta, "LicenseUrl"),
                "author": meta_value(meta, "Artist"),
                "attribution_required": meta_value(meta, "AttributionRequired") == "true",
                "date_taken": meta_value(meta, "DateTimeOriginal"),
                "description": meta_value(meta, "ImageDescription")[:200],
            })
        time.sleep(1)
    return pd.DataFrame(records)

protests = file_info(list(protest_files), found_in=protest_files)
print(f"{len(protests)} files with details")
protests[["title", "license", "author", "date_taken"]].head()

> **✏️ Exercise 2**
>
> Look at the licenses in the protest corpus.
>
> 1. How many files are there per license (`value_counts`)?
> 2. What share of the files require attribution, and how many are share-alike (`SA` in the
>    license name)?
> 3. If you published a figure with a grid of 20 of these images, what would you have to put in
>    the caption or an appendix?

In [ ]:
# ✅ Solution
print(protests["license"].value_counts(), "\n")

share_attrib = protests["attribution_required"].mean()
n_share_alike = protests["license"].str.contains("SA", case=True).sum()
print(f"Attribution required: {share_attrib:.0%} of files")
print(f"Share-alike licenses: {n_share_alike} files")

# 3. For every image: the author, the license (with a link) and the source page, plus a note if
#    you cropped or changed it. With 20 images that is too much for a caption, so a common
#    solution is a short caption ("Photos from Wikimedia Commons; credits in Appendix A") and a
#    table in the appendix generated straight from the manifest.

## 4. Downloading politely, with a manifest

Now we download the thumbnails. The function below follows the habits from module 01 (save to
disk, skip files you already have, pause between requests) and adds one more: it returns a
**manifest**, a table with one row per image that records where the file came from, under which
license, when you downloaded it and a checksum (SHA-256) that proves later that the file has not
changed. The manifest is the single most useful thing you can keep about a corpus.

Files are named after a short hash of their title, so that names are safe on every operating
system and two different titles can't clash.

In [ ]:
def safe_name(title, url):
    """A short, file-system-safe name based on the title, keeping the URL's file extension."""
    suffix = Path(urlparse(url).path).suffix.lower() or ".jpg"
    return hashlib.sha1(title.encode("utf-8")).hexdigest()[:12] + suffix

def download_images(table, folder, source, pause=0.5):
    """Download every `source_url` in `table` into `folder`; return the table as a manifest."""
    folder.mkdir(parents=True, exist_ok=True)
    rows = []
    for i, row in enumerate(table.to_dict("records")):
        path = folder / safe_name(row["title"], row["source_url"])
        status = "ok"
        if not path.exists():
            try:
                path.write_bytes(download(row["source_url"], retries=1, wait=5).content)  # one short retry per file
                time.sleep(pause)
            except requests.RequestException as error:
                status = f"failed: {error.__class__.__name__}"
        if status == "ok":
            data = path.read_bytes()
            with Image.open(io.BytesIO(data)) as img:
                width, height = img.size
            row.update(local_path=str(path), sha256=hashlib.sha256(data).hexdigest(), width=width, height=height)
        row.update(source=source, status=status, downloaded_at=datetime.now(timezone.utc).isoformat(timespec="seconds"))
        rows.append(row)
        if (i + 1) % 50 == 0:
            print(f"{i + 1} of {len(table)} done")
    manifest = pd.DataFrame(rows)
    if manifest.empty:      # nothing found: keep the columns, so later steps report zero files
        print(f"No files to download into {folder}.")
        manifest = pd.DataFrame(columns=[*table.columns, "local_path", "sha256", "width", "height", "source",
                                         "status", "downloaded_at"])
    return manifest

protest_manifest = download_images(protests, CORPUS_DIR / "protests", source="Wikimedia Commons")
print(protest_manifest["status"].value_counts())

In [ ]:
ok = protest_manifest[protest_manifest["status"] == "ok"]
contact_sheet(ok["local_path"].head(32), ok["found_in"].str.replace("Demonstrations and protests in the Netherlands in ", "").head(32))

Save the manifest next to the images. From now on, any analysis starts by reading the manifest,
not by listing a folder.

In [ ]:
protest_manifest.to_csv(CORPUS_DIR / "protests_manifest.csv", index=False)
protest_manifest[["local_path", "title", "license", "author", "sha256", "downloaded_at"]].head()

> **✏️ Exercise 3**
>
> Images in the protest categories are not all photos *of* protests: some show posters, police
> vans or empty squares. For a classifier you also need crowds that are **not** protests. Collect
> up to 50 such "hard negatives": photos of King's Day (*Koningsdag*), when Dutch streets fill with
> people in orange.
>
> Instead of a category, use the Commons **search** API: `list=search` with `srsearch` set to
> `"Koningsdag filetype:bitmap"`, `srnamespace` set to `6` (the namespace for files) and `srlimit`
> set to `50`. Then reuse `file_info` and `download_images` (save to `CORPUS_DIR / "koningsdag"`),
> save the result as `koningsdag_manifest.csv`, and show a contact sheet.

In [ ]:
# ✅ Solution
data = commons_api({"action": "query", "list": "search", "srsearch": "Koningsdag filetype:bitmap",
                    "srnamespace": 6, "srlimit": 50})
titles = [hit["title"] for hit in data["query"]["search"]
          if hit["title"].lower().endswith((".jpg", ".jpeg", ".png"))]
print(f"{len(titles)} search hits")

koningsdag = file_info(titles, found_in={t: "search: Koningsdag" for t in titles})
koningsdag_manifest = download_images(koningsdag, CORPUS_DIR / "koningsdag", source="Wikimedia Commons")
koningsdag_manifest.to_csv(CORPUS_DIR / "koningsdag_manifest.csv", index=False)

ok_k = koningsdag_manifest[koningsdag_manifest["status"] == "ok"]
contact_sheet(ok_k["local_path"].head(24), ok_k["license"].head(24))
# Search finds files whose title or description mentions the term, so expect some misses (a
# poster, a building, an orange cake). Like categories, search results need a look before use.

## 5. A second corpus: food packaging from Open Food Facts

Food packaging is a rich object for health communication research: it combines brand imagery,
cartoon characters, health claims and mandatory nutrition information on a few square
centimeters. Open Food Facts collects photos of products from volunteers, along with data such as
the **Nutri-Score** (a nutrition grade from A to E) and the product category.

Its search API lets us ask for products by category and country. Two rules from its
documentation: identify yourself with a User-Agent, and keep **search** requests to at most
10 per minute (product photos can be downloaded one at a time). Bulk users are asked to use the
data exports instead of the API.

The search service is also down fairly often: Open Food Facts' own
[status monitor](https://github.com/openfoodfacts/openfoodfacts-upptime) logs regular outages of
"Search API v2". Our `download` function waits and retries when a server answers that it is busy or
unavailable (status 429 or 5xx). If the search still fails, the cell below falls back to the course's
frozen product list, when the repository has one, or tells you to try again later. Outages like this
are one more reason to save a manifest of exactly what you collected.

In [ ]:
OFF_SEARCH = "https://world.openfoodfacts.org/api/v2/search"
OFF_FIELDS = "code,product_name,brands,image_front_url,nutriscore_grade,nova_group,labels_tags"

def off_search(category, country="netherlands", pages=2, page_size=100):
    """Search Open Food Facts for products in a category sold in a country."""
    products = []
    for page in range(1, pages + 1):
        data = download(OFF_SEARCH, params={
            "categories_tags_en": category, "countries_tags_en": country, "fields": OFF_FIELDS,
            "sort_by": "unique_scans_n", "page_size": page_size, "page": page}).json()
        products += data["products"]
        print(f"page {page}: {len(data['products'])} products (of {data['count']} in total)")
        if page * page_size >= data["count"]:
            break
        time.sleep(7)       # stay under 10 search requests per minute
    return pd.DataFrame(products)

FROZEN_PACKAGING = "https://raw.githubusercontent.com/denisejroth/pixels-frames-transformers/main/shared/data/packaging_manifest.csv"

try:
    cereals = off_search("breakfast-cereals")
except requests.HTTPError as error:
    print(f"The Open Food Facts search is not available right now ({error.response.status_code}).")
    try:
        cereals = pd.read_csv(io.StringIO(download(FROZEN_PACKAGING, retries=0).text), dtype={"code": str})
        print(f"Using the course's frozen list of {len(cereals)} products instead.")
    except requests.HTTPError:
        raise RuntimeError("Open Food Facts' search is down and the course has no frozen product list yet. "
                           "Wait a while (often an hour is enough) and run this cell again.") from None
cereals.head()

Not every product has a front photo. We keep those that do, add the license information (all
Open Food Facts photos are CC BY-SA, credited to the Open Food Facts contributors) and download
the 400-pixel versions with the same function as before.

In [ ]:
packages = cereals.dropna(subset=["image_front_url"]).copy()
packages["title"] = "OFF " + packages["code"].astype(str)
packages["source_url"] = packages["image_front_url"]
packages["page_url"] = "https://world.openfoodfacts.org/product/" + packages["code"].astype(str)
packages["license"] = "CC BY-SA"
packages["license_url"] = "https://world.openfoodfacts.org/terms-of-use"
packages["author"] = "Open Food Facts contributors"
packages["attribution_required"] = True

print(f"{len(packages)} of {len(cereals)} products have a front photo")
package_manifest = download_images(packages, CORPUS_DIR / "packaging", source="Open Food Facts", pause=1.0)
package_manifest.to_csv(CORPUS_DIR / "packaging_manifest.csv", index=False)
print(package_manifest["status"].value_counts())

In [ ]:
ok_p = package_manifest[package_manifest["status"] == "ok"]
contact_sheet(ok_p["local_path"].head(32), ok_p["product_name"].fillna("").head(32))

> **✏️ Exercise 4**
>
> 1. Show how many products fall in each Nutri-Score grade (`nutriscore_grade`). Which grades are
>    most common, and how many products have no grade?
> 2. Show a contact sheet of up to eight products with grade **a** and up to eight with grade **e**.
>    Do you see differences in the packaging (colors, characters, text)? Write down two features you
>    would want to measure. Module 03 will measure some of them.

In [ ]:
# ✅ Solution
print(ok_p["nutriscore_grade"].fillna("missing").value_counts(), "\n")

for grade in ["a", "e"]:
    subset = ok_p[ok_p["nutriscore_grade"] == grade].head(8)
    print(f"Nutri-Score {grade.upper()}: {len(subset)} shown")
    if len(subset):
        contact_sheet(subset["local_path"], subset["product_name"].fillna(""), n_cols=8)

# Grades in the middle tend to be most common for cereals, and some products have no grade
# ("unknown" or missing) because their nutrition data is incomplete. In the contact sheets,
# the e-grade products are often sweetened cereals with bright colors and cartoon characters,
# while a-grade products (oats, muesli, bran) tend to use calmer colors and photos of the grain.
# Features worth measuring: colorfulness and saturation, the presence of characters or faces,
# and the amount of text and health claims on the front.

## 6. Finding duplicates

Collections almost always contain duplicates: the same photo uploaded twice, a crop of another
image, or (on Open Food Facts) the same packaging under several barcodes. Duplicates inflate
your counts and, worse, can end up in both your training and test data, which makes a classifier
look better than it is.

**Exact** duplicates are easy: identical files have the same SHA-256 checksum, which we already
stored.

In [ ]:
manifest = pd.concat([protest_manifest, package_manifest], ignore_index=True)
manifest = manifest[manifest["status"] == "ok"].reset_index(drop=True)

exact = manifest[manifest.duplicated("sha256", keep=False)].sort_values("sha256")
print(f"{len(exact)} files share a checksum with another file")
exact[["source", "title", "sha256"]].head(10)

**Near** duplicates (resized, recompressed, slightly cropped or brightened copies) have
completely different checksums, because a checksum changes if a single bit changes. A
**perceptual hash** works differently: it shrinks the image to a tiny grayscale version and
summarizes its overall structure in 64 bits. Similar-looking images get similar hashes, and the
number of bits that differ (the Hamming distance, from 0 to 64) measures how different they look.

Let's see how a perceptual hash reacts when we change one image in different ways.

In [ ]:
original = Image.open(manifest.loc[0, "local_path"]).convert("RGB")
w, h = original.size

def recompress(img, quality):
    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=quality)
    return Image.open(io.BytesIO(buffer.getvalue()))

variants = {
    "resized to 50%": original.resize((w // 2, h // 2)),
    "JPEG quality 20": recompress(original, 20),
    "brighter": Image.eval(original, lambda v: min(255, int(v * 1.3))),
    "cropped 10%": original.crop((w // 20, h // 20, w - w // 20, h - h // 20)),
    "mirrored": ImageOps.mirror(original),
    "a different image": Image.open(manifest.loc[len(manifest) - 1, "local_path"]).convert("RGB"),
}

base_hash = imagehash.phash(original)
print(f"{'variant':<20} {'checksum same?':<16} perceptual distance")
for name, img in variants.items():
    same_checksum = hashlib.sha256(img.tobytes()).digest() == hashlib.sha256(original.tobytes()).digest()
    print(f"{name:<20} {str(same_checksum):<16} {imagehash.phash(img) - base_hash}")

> **✏️ Exercise 5**
>
> 1. Add two more variants of your own to the comparison, for example a **grayscale** version and a
>    version **rotated by 5 degrees**. How far do they move the hash?
> 2. Based on all variants, what threshold (maximum distance) would you use to call two images
>    "near duplicates"? What kind of copy would your threshold miss?

In [ ]:
# ✅ Solution
more = {
    "grayscale": original.convert("L").convert("RGB"),
    "rotated 5 degrees": original.rotate(5, expand=False, fillcolor=(0, 0, 0)),
    "cropped 25%": original.crop((w // 8, h // 8, w - w // 8, h - h // 8)),
}
for name, img in {**variants, **more}.items():
    print(f"{name:<20} {imagehash.phash(img) - base_hash}")

# Resizing, recompression, brightening and grayscale barely move the hash (usually 0 to 4 bits).
# Small crops and rotations move it more, mirrored images and different images land far away
# (often 25 to 35 bits). A threshold of about 8 to 10 catches the usual re-uploads and resized
# copies without many false alarms. It misses heavier crops, rotations and mirrored copies; if
# those matter, compare the image embeddings from module 06 instead.

Now we compute the hash of every image and look for pairs that are close. With a few hundred
images we can simply compare every pair: the hashes become rows of 0s and 1s, and the distance is
the number of positions where two rows differ.

In [ ]:
hashes = [imagehash.phash(Image.open(p).convert("RGB")) for p in manifest["local_path"]]
bits = np.array([h.hash.flatten() for h in hashes], dtype=np.uint8)
distance = (bits[:, None, :] != bits[None, :, :]).sum(axis=2)
np.fill_diagonal(distance, 64)          # ignore each image's distance to itself

i_idx, j_idx = np.where(np.triu(distance <= 10, k=1))
pairs = sorted(zip(distance[i_idx, j_idx], i_idx, j_idx))
print(f"{len(pairs)} pairs within distance 10")

for d, i, j in pairs[:6]:
    fig, axes = plt.subplots(1, 2, figsize=(4, 2.2))
    for ax, k in zip(axes, [i, j]):
        ax.imshow(Image.open(manifest.loc[k, "local_path"]).convert("RGB"))
        ax.set_title(manifest.loc[k, "title"][:24], fontsize=7)
        ax.axis("off")
    fig.suptitle(f"distance {d}", fontsize=9)
    plt.show()

> **✏️ Exercise 6**
>
> Mark the near duplicates in the manifest so later analyses can drop them.
>
> 1. Group the images so that any two images within distance 10 of each other end up in the same
>    group (if A is close to B and B is close to C, all three form one group).
> 2. Add a column `duplicate_group` with a group number, and a column `keep` that is `True` for one
>    image per group (say, the largest file) and for every image without duplicates.
> 3. How many images do you keep?
>
> *(Hint: a simple way to build the groups is to give every image its own group number, then
> repeatedly merge the groups of every close pair until nothing changes.)*

In [ ]:
# ✅ Solution
group = np.arange(len(manifest))
changed = True
while changed:
    changed = False
    for d, i, j in pairs:
        low = min(group[i], group[j])
        if group[i] != low or group[j] != low:
            group[group == group[i]] = low
            group[group == group[j]] = low
            changed = True

manifest["duplicate_group"] = group
manifest["file_size"] = [Path(p).stat().st_size for p in manifest["local_path"]]
largest = manifest.groupby("duplicate_group")["file_size"].idxmax()
manifest["keep"] = manifest.index.isin(largest)

print(f"Keeping {manifest['keep'].sum()} of {len(manifest)} images")
print(manifest.groupby("source")["keep"].agg(["sum", "count"]))
# Keeping the largest file is one sensible rule (most detail); keeping the oldest upload is
# another (closest to the original). Whatever you choose, write it down: it is a decision about
# your data, just like a preprocessing step for text.

## 7. What you can share, and how to document it

When you publish, reviewers and readers will want to see your data. With images you often can't
share the files themselves, but you can almost always share enough to rebuild the corpus:

- the **manifest**: identifiers and URLs, license, author, the date you downloaded each file,
  and its checksum, so others can check they have the same file;
- your **code** for collecting, filtering and deduplicating;
- **derived data** that is not the image itself: features, embeddings, codes and labels.

For an open corpus like ours, sharing the files is allowed as long as the manifest travels with
them, and the CC BY-SA images keep their license.

A **datasheet** (Gebru et al., 2021) is a short document that answers standard questions about a
dataset: why it was made, what it contains, how it was collected, what was left out and why, and
what it should and shouldn't be used for. Writing one takes half an hour and saves your future
self, and your readers, much more.

> **✏️ Exercise 7**
>
> Write a short datasheet (5 to 10 sentences, or bullet points) for the protest corpus you built in
> this notebook. Cover at least:
>
> - **Motivation:** what is the corpus for?
> - **Composition:** what does it contain, how many images, from when and where?
> - **Collection:** how were images selected, and what was deliberately skipped?
> - **Known gaps and biases:** what kinds of protests or people might be over- or
>   under-represented, and why?
> - **Uses:** what should it *not* be used for?

**✅ One possible answer**

- **Motivation:** a small teaching corpus for learning to code and classify protest images, not a
  representative sample of Dutch protest.
- **Composition:** about 300 JPEG and PNG photos (500-pixel thumbnails) from the Wikimedia
  Commons categories "Demonstrations and protests in the Netherlands in 2015" to "... in 2024" and
  one level of their subcategories, plus up to 50 King's Day photos as non-protest crowds. Each
  file has an author, a free license and a source page in the manifest.
- **Collection:** collected through the Commons API on the download date in the manifest.
  Subcategories with words suggesting nudity were skipped, and videos, PDFs and drawings were
  excluded. Near duplicates (perceptual hash distance of 10 or less) are marked, with the largest
  file kept.
- **Known gaps and biases:** Commons reflects what volunteers photographed and uploaded, so large
  and well-known events in big cities, and photographers' tastes, are over-represented; small,
  local or quickly dispersed protests are rare. Categories are assigned by volunteers and contain
  some images that don't show a protest at all. Recent years dominate.
- **Uses:** fine for teaching and for testing methods. Not suitable for estimating how often or
  where protests happen, and not for identifying individual participants: people in these
  photos did not consent to being studied, so faces should not be analyzed or published in
  identifiable form.

## Wrap-up

You have seen that:

- the questions about licenses, people, platform terms and sharing come before the first
  download, because they decide what you can do with the corpus;
- open sources like Wikimedia Commons and Open Food Facts give you a license and an author for
  every file through their APIs;
- polite collection means identifying yourself, pausing between requests and never downloading
  the same file twice;
- a manifest with source, license, download date and checksum is the backbone of a corpus;
- checksums find exact duplicates, and perceptual hashes find resized or recompressed copies;
- you can usually share enough to rebuild a corpus even when you can't share the images, and a
  datasheet tells others what your corpus is and isn't good for.

**Next:** in module 03 you measure color, light, faces and text in these images, and check
whether those measurements mean what you think they mean.